In [1]:
import os
import zipfile  
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import pandas as pd
import timm
from PIL import Image
from io import BytesIO

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from pathlib import Path

DATA_DIR = Path("/kaggle/input/competitions/siim-isic-melanoma-classification")
CSV_PATH = DATA_DIR / "train.csv"
IMAGE_DIR = DATA_DIR / "jpeg/train"

print("CSV exists:", CSV_PATH.exists())
print("Image folder exists:", IMAGE_DIR.exists())
print("CUDA available:", torch.cuda.is_available())

CSV exists: True
Image folder exists: True
CUDA available: True


In [2]:
df = pd.read_csv(CSV_PATH)  # NEW: load Kaggle training metadata

print(df.head())
print(df.columns)
print("\nClass distribution:")
print(df["target"].value_counts())

     image_name  patient_id     sex  age_approx anatom_site_general_challenge  \
0  ISIC_2637011  IP_7279968    male        45.0                     head/neck   
1  ISIC_0015719  IP_3075186  female        45.0               upper extremity   
2  ISIC_0052212  IP_2842074  female        50.0               lower extremity   
3  ISIC_0068279  IP_6890425  female        45.0                     head/neck   
4  ISIC_0074268  IP_8723313  female        55.0               upper extremity   

  diagnosis benign_malignant  target  
0   unknown           benign       0  
1   unknown           benign       0  
2     nevus           benign       0  
3   unknown           benign       0  
4   unknown           benign       0  
Index(['image_name', 'patient_id', 'sex', 'age_approx',
       'anatom_site_general_challenge', 'diagnosis', 'benign_malignant',
       'target'],
      dtype='object')

Class distribution:
target
0    32542
1      584
Name: count, dtype: int64


In [3]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["target"],
    random_state=42
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

# CHANGED: modest oversampling (5:1 ratio) instead of full balance
# Full balance (56x repetition) causes the model to memorise minority samples
benign_df   = train_df[train_df["target"] == 0]
melanoma_df = train_df[train_df["target"] == 1]

target_minority = len(benign_df) // 5  # 5:1 ratio

melanoma_upsampled = melanoma_df.sample(
    n=target_minority,
    replace=True,
    random_state=42
)

train_df_balanced = pd.concat([benign_df, melanoma_upsampled], axis=0)
train_df_balanced = train_df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print("Original train size:",  len(train_df))
print("Balanced train size:",  len(train_df_balanced))
print("Validation size:",      len(val_df))
print("\nBalanced training class distribution:")
print(train_df_balanced["target"].value_counts())
print("\nValidation class distribution:")
print(val_df["target"].value_counts())

Original train size: 26500
Balanced train size: 31239
Validation size: 6626

Balanced training class distribution:
target
0    26033
1     5206
Name: count, dtype: int64

Validation class distribution:
target
0    6509
1     117
Name: count, dtype: int64


In [4]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(25),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomAffine(degrees=10, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.15)),  # Must be after ToTensor
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [5]:
class MelanomaDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        image_name = row["image_name"]
        label = int(row["target"])

        image_path = self.image_dir / f"{image_name}.jpg"
        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return image, label

In [6]:
USE_EXTRACTED_IMAGES = IMAGE_DIR.exists()  # NEW: decide whether to use extracted folder

if USE_EXTRACTED_IMAGES:
    print("Using extracted JPEGs from:", IMAGE_DIR)  # NEW
    train_set = MelanomaDataset(train_df_balanced, IMAGE_DIR, transform=train_transform)  # CHANGED: use balanced training set
    val_set = MelanomaDataset(val_df, IMAGE_DIR, transform=val_transform)

else:
    print("Using images directly from zip:", ZIP_PATH)  # NEW
    train_set = MelanomaDataset(train_df, zip_path=ZIP_PATH, transform=train_transform)
    val_set = MelanomaDataset(val_df, zip_path=ZIP_PATH, transform=val_transform)

Using extracted JPEGs from: /kaggle/input/competitions/siim-isic-melanoma-classification/jpeg/train


In [7]:
train_loader = DataLoader(
    train_set,
    batch_size=64,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_set,
    batch_size=64,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Train batches: 489
Validation batches: 104


In [8]:
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("First 10 labels:", labels[:10])

Image batch shape: torch.Size([64, 3, 224, 224])
Label batch shape: torch.Size([64])
First 10 labels: tensor([0, 0, 0, 0, 1, 0, 0, 0, 0, 1])


In [9]:
# CHANGED: EfficientNet-B3 via timm instead of ResNet18
# EfficientNet-B3 has stronger feature extraction and ~5x more capacity
# Change model_name to 'efficientnet_b4' or 'vit_small_patch16_224' to experiment

class MelanomaModel(nn.Module):
    def __init__(self, model_name='efficientnet_b3', n_classes=2):
        super().__init__()
        self.backbone = timm.create_model(
            model_name, pretrained=True, num_classes=0, global_pool='avg'
        )
        in_features = self.backbone.num_features
        self.head = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(256, n_classes)
        )

    def forward(self, x):
        return self.head(self.backbone(x))

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # NEW
model = MelanomaModel(n_classes=2).to(device)  # NEW

print("Using device:", device)

Using device: cuda


In [11]:
from torch import optim

NUM_EPOCHS    = 15
LEARNING_RATE = 1e-4  # CHANGED: lower LR suits pretrained EfficientNet

optimizer = optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5
)

# CHANGED: CosineAnnealingLR decays LR smoothly — almost always improves final AUC
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-6
)

# Weighted loss to handle remaining class imbalance
class_counts = train_df_balanced["target"].value_counts().sort_index()
weights = torch.tensor(
    [1.0 / class_counts[0], 1.0 / class_counts[1]], dtype=torch.float32
)
weights = weights / weights.sum() * 2
criterion = nn.CrossEntropyLoss(weight=weights.to(device))

print("Class counts:\n", class_counts)
print("Class weights:", weights)

Class counts:
 target
0    26033
1     5206
Name: count, dtype: int64
Class weights: tensor([0.3333, 1.6667])


In [12]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0

    for images, labels in loader:
        optimizer.zero_grad()

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)
        loss    = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(loader)

In [13]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score

def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0

    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            probs = torch.softmax(outputs, dim=1)[:, 1]
            preds = torch.argmax(outputs, dim=1)

            running_loss += loss.item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    val_loss = running_loss / len(loader)
    val_acc = accuracy_score(all_labels, all_preds)

    return val_loss, val_acc, all_labels, all_preds, all_probs

In [ ]:
train_losses, val_losses, val_accuracies, val_aucs = [], [], [], []
best_auc = 0.0

for epoch in range(NUM_EPOCHS):
    train_loss                          = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc, y_true, y_pred, y_prob = validate(model, val_loader, criterion, device)

    # CHANGED: step the LR scheduler every epoch
    scheduler.step()

    # CHANGED: track AUC — accuracy alone is misleading with imbalanced data
    val_auc = roc_auc_score(y_true, y_prob)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)
    val_aucs.append(val_auc)

    print(
        f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val AUC: {val_auc:.4f} | "
        f"LR: {scheduler.get_last_lr()[0]:.2e}"
    )

    # CHANGED: save the best model by AUC, not just at the end
    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), "best_model_efficientnet_b3.pth")
        print(f"  -> New best AUC: {best_auc:.4f} — model saved.")

print(f"\nTraining complete. Best Val AUC: {best_auc:.4f}")

In [ ]:
# Reload best checkpoint for final evaluation
model.load_state_dict(torch.load("best_model_efficientnet_b3.pth", map_location=device))
_, _, y_true, y_pred, y_prob = validate(model, val_loader, criterion, device)

print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=["Benign", "Melanoma"]))

cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:\n", cm)
print("ROC-AUC:", roc_auc_score(y_true, y_prob))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confusion matrix
axes[0].imshow(cm, cmap="Blues")
axes[0].set_title("Confusion Matrix")
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(["Benign", "Melanoma"])
axes[0].set_yticks([0, 1]); axes[0].set_yticklabels(["Benign", "Melanoma"])
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        axes[0].text(j, i, cm[i, j], ha="center", va="center", color="black")

# Loss curves
axes[1].plot(train_losses, label="Train Loss")
axes[1].plot(val_losses,   label="Val Loss")
axes[1].set_title("Loss Curves")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# CHANGED: plot AUC curve over epochs (not just loss)
plt.figure(figsize=(8, 5))
plt.plot(val_aucs, marker='o', label="Val AUC")
plt.axhline(y=max(val_aucs), color='r', linestyle='--', alpha=0.5, label=f"Best AUC: {max(val_aucs):.4f}")
plt.xlabel("Epoch"); plt.ylabel("AUC-ROC")
plt.title("Validation AUC-ROC over Training")
plt.legend(); plt.grid(True, alpha=0.3)
plt.ylim([0.5, 1.0])
plt.show()

In [ ]:
# Final model already saved at best checkpoint during training
print(f"Best model saved to: best_model_efficientnet_b3.pth")
print(f"Best Val AUC: {best_auc:.4f}")